In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.health_metrics AS (
    SELECT
        b.lead_id,
        b.status_label,
        b.date_updated,
        -- Real client name: Calendly notifications quote the invitee's real
        -- name ("Invitee: <name>") in body_text_quoted — the only unmasked
        -- name source in the data (the CRM masks every person-name field).
        -- Hash fallback when the client never booked a meeting. Coverage
        -- 843/1,380 population leads (verified live, Nov 11 2026).
        COALESCE(inv.real_name, b.lead_name) AS lead_name,
        b.lead_display_name,
        le.customer_emails,
        le.primary_customer_email,
        b.csm_id,
        csm.csm_name,
        csm.csm_email,
        b.Contracted_Value,
        b.Tier,
        b.Avatar,
        sd.date_of_sale,
        cs.Contract_Start_Date AS contract_start_date,
        sd.program,
        sd.setter_name,
        sd.closer_name,
        sd.contract_value AS sale_contract_value,
        e.days_since_last_email,
        m.days_since_last_meeting,
        COALESCE(mc.missed_calls_14_days, 0) AS missed_calls_14_days,
        nm.days_until_next_meeting,
        l.days_since_last_login,
        cl.days_since_last_meeting_calendly,
        s.sentiment,
        s.days_since_last_message_from_client,
        s.days_since_last_message_from_team_member,
        (sh.staff_hash IS NOT NULL) AS is_intern_client,
        (spine.h IS NULL) AS identity_unlinked,
        (le.customer_emails IS NULL OR size(le.customer_emails) = 0) AS no_client_email
    FROM crm_ingestion.gold.lead_dim b
    LEFT JOIN crm_ingestion.gold.lead_emails le ON b.lead_id = le.lead_id
    -- REAL CLIENT NAME — resolved from the Calendly invitee lines quoted in
    -- email notification bodies. Distinct non-hash invitee values per lead,
    -- sorted, joined with " / " (the reference profile's LEAD_NAME format).
    -- Verified on traced leads: reproduces the reference exactly for
    -- Weed/Frost/Sarumathi; LeValle gains a nickname variant; leads that
    -- never booked keep the identity hash. Identity joins below (staff_hash,
    -- spine) still use the ORIGINAL b.lead_name hash — do not repoint them.
    LEFT JOIN (
        SELECT lead_id,
               ARRAY_JOIN(SORT_ARRAY(COLLECT_SET(DISTINCT name_val)), ' / ') AS real_name
        FROM (
            SELECT lead_id,
                   TRIM(REGEXP_EXTRACT(CAST(body_text_quoted AS STRING),
                       '(?s)Invitee:\s*\n+\s*([^\n]+)', 1)) AS name_val
            FROM crm_ingestion.silver.lead_activites_processed
            WHERE CAST(body_text_quoted AS STRING) LIKE '%Invitee:%'
        )
        WHERE name_val != '' AND name_val NOT LIKE 'user\_%'
        GROUP BY lead_id
    ) inv ON inv.lead_id = b.lead_id
    LEFT JOIN crm_ingestion.gold.sales_fact sd ON b.lead_id = sd.lead_id
    -- Contract start from the CRM's stored lead custom field (latest snapshot per
    -- lead). Matches the reference output exactly (verified 3/3 on traced leads);
    -- covers 1,334 of 1,371 clients (37 residual = no custom field and no
    -- qualifying sale activity — verified source gap, not a pipeline issue). The sale-activity date stays available
    -- separately as date_of_sale — they disagree 141/172 where both exist.
    LEFT JOIN (
        SELECT leads_processed_id AS lead_id, Contract_Start_Date
        FROM crm_ingestion.silver.leads_processed_custom
        QUALIFY ROW_NUMBER() OVER (PARTITION BY leads_processed_id
                                   ORDER BY bronze_insert_date DESC) = 1
    ) cs ON cs.lead_id = b.lead_id
    LEFT JOIN crm_ingestion.gold.staff_dim csm ON b.csm_id = csm.csm_id
    LEFT JOIN crm_ingestion.gold.days_since_last_email e ON b.lead_id = e.lead_id
    LEFT JOIN crm_ingestion.gold.days_since_last_meeting m ON b.lead_id = m.lead_id
    LEFT JOIN crm_ingestion.gold.missed_calls_14_days mc ON b.lead_id = mc.lead_id
    LEFT JOIN crm_ingestion.gold.days_until_next_meeting nm ON b.lead_id = nm.lead_id
    LEFT JOIN crm_ingestion.gold.days_since_last_login l ON b.lead_id = l.lead_id
    LEFT JOIN crm_ingestion.gold.days_since_last_meeting_calendly cl ON b.lead_id = cl.lead_id
    LEFT JOIN crm_ingestion.gold.sentiment s ON b.lead_id = s.lead_id
    -- INTERN CLIENT: the lead's own identity hash belongs to a staff person —
    -- a former client who now works here. The client-to-coach conversion is
    -- one-way: once staff, never a regular client again. They stay in this
    -- table, flagged, for tracking; 3_14 excludes them from the client book.
    LEFT JOIN (
        SELECT DISTINCT explode(user_32digits) AS staff_hash
        FROM crm_ingestion.gold.staff_emails
    ) sh ON sh.staff_hash = b.lead_name
    -- ATTENTION FLAGS — issues that need eyes, each on its own column:
    --   identity_unlinked: the lead's hash exists nowhere in the identity
    --     spine — client or staff cannot be proven. Includes the 11
    --     staff-contact-card leads ($60K contracts) that might be intern
    --     clients in disguise. Needs human review.
    --   no_client_email: every email ever attached to this lead is
    --     staff-owned (or none exists) — unreachable by email today.
    LEFT JOIN (
        SELECT DISTINCT explode(user_32digits) AS h
        FROM crm_ingestion.silver.user_identity_map
    ) spine ON spine.h = b.lead_name
)